In [1]:
#import packages
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pymannkendall as mk
import cartopy.crs as ccrs
import time
import matplotlib as mpl
from scipy.stats import linregress
import dask
import matplotlib.ticker as mticker
import string
import os
letters=[]
for letter in string.ascii_lowercase[0:26]:
    letters.append(letter+". ")
from functions import *
for letter in string.ascii_lowercase:
    letters.append(letter+". ")

era_land=xr.open_dataset('eralandmask_regid.nc').rename({'lat':'latitude','lon':'longitude'})

In [3]:
#functions
from functions import *
def calculate_anomaly(ds):#calculate anomally

    gb = ds.groupby("time.dayofyear")
    clim = gb.mean(dim='time')
    return (gb - clim).drop_vars('dayofyear')
def refqtl_single2(dat, p=0.9): #Calculate reference quantiles
    years=np.arange(np.unique(dat.time.dt.year)[0], np.unique(dat.time.dt.year)[30]) #define reference clim years
    ref_t=dat.sel(time=dat.time.dt.year.isin(years))
    doys=np.arange(1,366)
    #subset data for quantile quantile across rolling dimension
    #     out_qtl=xr.full_like(ref_t.isel(time=0), fill_value=np.nan)
    #     out_qtl=out_qtl.expand_dims(time=np.arange(1,366)).copy(deep=True) #define doys for output
    #     out_qtl=out_qtl.rename({'time':'dayofyear'})
    results=[]
    doy_dat = ref_t.time.dt.dayofyear
    for i in range(365):
        if i<8:
            doy1=doys[i:i+8]
            doy2=doys[-i-7:]
            curdoy=np.concatenate((doy1,doy2))
        elif i>357:
            doy1=doys[i-7:]
            doy2=doys[0:i-357]
            curdoy=np.concatenate((doy1,doy2))
        else:
            curdoy=doys[i-7:i+8]
        ref_doy=ref_t.sel(time=doy_dat.isin(curdoy))
        #quantile along single doy time
        ref_ptl=ref_doy.quantile(0.9,dim='time')
        results.append(ref_ptl)
    out_qtl=xr.concat(results, dim='dayofyear')
    out_qtl=out_qtl.assign_coords(dayofyear=np.arange(1,366))

    return out_qtl.drop_vars('quantile')
def thresh(ds,qt):#Identify days exceeding quantiles

    thr = qt.sel(dayofyear=ds.time.dt.dayofyear)
    thresh_alt = ds > thr
    try:
        thresh_alt=thresh_alt.drop_vars('dayofyear')
    except:
        pass
    return thresh_alt
def hotspellsfix(ds):#Identify heatwave days (excceeding quantiles for 3+ consecutive days)
    N = 3

    # Step 1: find windows where all N values are True
    mask = ds.rolling(time=N, min_periods=N).sum() == N

    # Step 2: expand those windows across the full run
    heat_days = xr.concat(
        [mask.shift(time=-i) for i in range(N)],
        dim="expand"
    ).any("expand")

    # Step 3: clean up edges 
    heat_days = heat_days.fillna(False) & ds

    # Optional: convert to int
    heat_daysout2 = heat_days.astype(int)
    return heat_daysout2
def seas_change(ds,yrs):#Change in hot spell days against reference period
    heat_seas = ds.groupby("time.year").map(
        lambda x: x.groupby("time.season").sum("time")
    )
    ref = heat_seas.sel(year=slice(1950,1980)).mean("year")

    heat_change = heat_seas.sel(year=slice(yrs[0],yrs[1])).mean("year") - ref
    try:
        heat_change=heat_change.drop_vars('time')
    except:
        pass
    return heat_change 



### CanDCS

In [7]:
#re open tasmean files (CanDCS)
filenames = sorted(os.listdir('tmean/.'))
models_mn=[]
to_con_mn=[]
for entry in filenames:
    if entry.startswith("tasmean_day"):
        #models.append(entry[27:-42])
        dat=xr.open_dataset('tmean/'+entry)

        dat1=dat.rename({'lon':'longitude','lat':'latitude'}) #.convert_calendar('noleap')
        temp_dat=rotlon_180(dat1)
        tmp1=temp_dat.sortby(temp_dat.time)
        tmp_midl=cutmidlat(tmp1)
        tmp_midlat=landmask(tmp_midl)
        tmp_midlat=tmp_midlat.drop_duplicates(dim="time")
        #temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2021-12-31')))
        try:
            to_con_mn.append(tmp_midlat.convert_calendar('noleap'))
            models_mn.append(entry[28:-13])#.sel(time=(slice('1950-01','2012-12'))))
        except:
            pass

In [8]:
#compute hotspells fixed # low resolution CanDCS
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=20, threads_per_worker=1)
client = Client(cluster)
for i,mod in enumerate(to_con_mn):
    #prep data
    #temp_dat1=xr.open_dataset('native/tasmax_day_MBCn+PCIC-Blend_'+mod+'_historical+ssp585_regridcon.nc', chunks={'lat':10,'lon':10,'time':-1})#.load()
    temp_dat2=mod.tas.to_dataset(name='heatdays').chunk({'latitude':10,'longitude':10,'time':-1})

    # anomally
    anom=xr.map_blocks(calculate_anomaly,temp_dat2,template=temp_dat2)

    #quantiles
    template1=anom.isel(time=slice(0,365)).rename({'time':'dayofyear'})
    template1['dayofyear']=template1.dayofyear.dt.dayofyear
    #dmaxanom=cuttimestart(dmaxanom,1979)
    outqt_anom= xr.map_blocks(refqtl_single2,anom, template=template1)

    #thresholds
    thresh_alt= xr.map_blocks(thresh,anom,args=([outqt_anom]), template=anom)

    #hsds
    heat_days= xr.map_blocks(hotspellsfix,thresh_alt, template=anom)

    heat_days= heat_days.load()
    ds_heat = xr.Dataset(
    data_vars=dict(
        heatdays=(["time", "latitude", "longitude"], heat_days.heatdays.values),
    ),
    coords=dict(
        longitude=("longitude", heat_days.longitude.values),
        latitude=("latitude", heat_days.latitude.values),
        time=("time", heat_days.time.values),
    ),
    attrs=dict(description="CanDCS hot spell data"),)
    
    ds_heat.to_netcdf('heat/'+models_mn[i]+'_tasmean_heat_days_1950-2100_fixednew.nc')

# for i,mod in enumerate(to_con_mn[12:]):
#     temp=mod.tas.to_dataset(name='t2m')
#     dmaxanom=temp.groupby('time.dayofyear')-temp.groupby('time.dayofyear').mean(dim='time')
#     dmaxanom=dmaxanom.sortby(dmaxanom.time)
#     #dmaxanom=cuttimestart(dmaxanom,1979)
#     outqt_anom=refqtl_single(dmaxanom.drop_vars('dayofyear'))
#     #outqt_anom.to_netcdf('cmip6/'+mod+'_anom_quantiles_15dsmooth_1950_2023_1x1.nc')

#     thresh_anom=thresh_single(dmaxanom.drop_vars('dayofyear'),outqt_anom)
#     thresh_anom=thresh_anom.sortby(thresh_anom.time)
#     #thresh_anom.to_netcdf('cmip6/'+mod+'_anom_thresholds_5dsmooth_1950_2023_1x1.nc')

#     heat_1_d=heat_day1(thresh_anom,dmaxanom,pst=3)
#     heat_1_d.to_dataset(name='htdays').to_netcdf('heat/'+models_mn[12+i]+'_tmean_heatdays_hist_1950_2100_regridcon.nc')

/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/distributed/client.py:3108: UserWarning: Sending large graph of size 146.25 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/distributed/worker.py:2975: UserWarning: Large object of size 3.05 MiB detected in task graph: 
  ([('time',), <xarray.IndexVariable 'time' (time: 5 ... 'axis': 'T'}],)
Consider scattering large objects ahead of time
with client.scatter to reduce scheduler burden and 
keep data on workers

    future = client.submit(func, big_data)    # bad

    big_future = client.scatter(big_data)     # good
    future = client.submit(func, big_future)  # good
  warnings.warn(
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, 

2026-04-10 17:43:05,133 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,149 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,412 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,498 - distributed.utils_perf - WARNING - full garbage collections took 23% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,533 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,619 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,667 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:05,821 - distributed.utils_perf - WARNING - full garbage collections took

2026-04-10 17:43:15,052 - distributed.utils_perf - WARNING - full garbage collections took 29% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,397 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,446 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,459 - distributed.utils_perf - WARNING - full garbage collections took 14% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,526 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,612 - distributed.utils_perf - WARNING - full garbage collections took 14% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,616 - distributed.utils_perf - WARNING - full garbage collections took 31% CPU time recently (threshold: 10%)
2026-04-10 17:43:15,896 - distributed.utils_perf - WARNING - full garbage collections took

2026-04-10 17:43:45,640 - distributed.utils_perf - WARNING - full garbage collections took 27% CPU time recently (threshold: 10%)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
2026-04-10 17:43:45,767 - distributed.utils_perf - WARNING - full garbage collections took 29% CPU time recently (threshold: 10%)
2026-04-10 17:43:45,884 - distributed.utils_perf - WARNING - full garbage collections took 14% CPU time recently (threshold: 10%)
2026-04-10 17:43:45,902 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:43:46,037 - distributed.utils_perf - WARNING - full garbage collections took 14% CPU time recently (threshold: 10%)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountere

2026-04-10 17:43:55,074 - distributed.utils_perf - WARNING - full garbage collections took 12% CPU time recently (threshold: 10%)
2026-04-10 17:43:55,420 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:43:55,569 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:43:55,999 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:43:56,139 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:56,219 - distributed.utils_perf - WARNING - full garbage collections took 17% CPU time recently (threshold: 10%)
2026-04-10 17:43:56,292 - distributed.utils_perf - WARNING - full garbage collections took 28% CPU time recently (threshold: 10%)
2026-04-10 17:43:56,646 - distributed.utils_perf - WARNING - full garbage collections took

...

2026-04-10 17:55:41,797 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-04-10 17:55:43,673 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/distributed/client.py:3108: UserWarning: Sending large graph of size 146.25 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
2026-04-10 17:55:48,225 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-04-10 17:55:50,165 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-04-10 17:55:52,019 - distributed.utils_perf - WARNING - full garbage collections took 12% CPU time recently (threshold: 10%)
2026-04-10 17:55:53,822 - distributed.utils_perf - WARNING - full garbage collections took 1

2026-04-10 17:56:00,611 - distributed.utils_perf - WARNING - full garbage collections took 21% CPU time recently (threshold: 10%)
2026-04-10 17:56:00,709 - distributed.utils_perf - WARNING - full garbage collections took 19% CPU time recently (threshold: 10%)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
2026-04-10 17:56:01,651 - distributed.utils_perf - WARNING - full garbage collections took 21% CPU time recently (threshold: 10%)
2026-04-10 17:56:01,796 - distributed.utils_perf - WARNING - full garbage collections took 19% CPU time recently (threshold: 10%)
2026-04-10 17:56:01,797 - distributed.utils_perf - WARNING - full ga

2026-04-10 17:56:37,020 - distributed.utils_perf - WARNING - full garbage collections took 23% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,027 - distributed.utils_perf - WARNING - full garbage collections took 18% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,035 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,041 - distributed.utils_perf - WARNING - full garbage collections took 19% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,054 - distributed.utils_perf - WARNING - full garbage collections took 15% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,056 - distributed.utils_perf - WARNING - full garbage collections took 20% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,081 - distributed.utils_perf - WARNING - full garbage collections took 21% CPU time recently (threshold: 10%)
2026-04-10 17:56:37,180 - distributed.utils_perf - WARNING - full garbage collections took

2026-04-10 17:56:41,557 - distributed.utils_perf - WARNING - full garbage collections took 20% CPU time recently (threshold: 10%)
2026-04-10 17:56:41,608 - distributed.utils_perf - WARNING - full garbage collections took 17% CPU time recently (threshold: 10%)
2026-04-10 17:56:41,632 - distributed.utils_perf - WARNING - full garbage collections took 23% CPU time recently (threshold: 10%)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1577: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
2026-04-10 17:56:41,893 - distributed.utils_perf - WARNING - full garbage collections took 20% CPU time recently (threshold: 10%)
2026-04-10 17:56:42,149 - distributed.utils_perf - WARNING - full ga

2026-04-10 17:56:51,980 - distributed.utils_perf - WARNING - full garbage collections took 25% CPU time recently (threshold: 10%)
2026-04-10 17:58:20,141 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)


In [7]:
#read pcic blend

tmn_pc=xr.open_dataset('obs_tasmax_day_PCIC_Blended_Observations_v1_1950-2012_regridcon.nc')
tmn_pc1=tmn_pc.sortby(tmn_pc.time).rename({'lat':'latitude','lon':'longitude'})
#heat79=cuttimestart(heat1,1979)
tmnpc_midl=cutmidlat(rotlon_180(tmn_pc1))

tmnpc_midlat=landmask(tmnpc_midl)
tmnpc_noleap=tmnpc_midlat.convert_calendar('noleap').drop_duplicates(dim="time")
#


In [9]:
#compute hotspell blend fixed


temp_dat2=tmnpc_noleap.tasmax.to_dataset(name='heatdays').chunk({'latitude':10,'longitude':10,'time':-1})
# temp_dat2=mod

# anomally
anom=xr.map_blocks(calculate_anomaly,temp_dat2,template=temp_dat2)

#quantiles
template1=anom.isel(time=slice(0,365)).rename({'time':'dayofyear'})
template1['dayofyear']=template1.dayofyear.dt.dayofyear
#dmaxanom=cuttimestart(dmaxanom,1979)
outqt_anom= xr.map_blocks(refqtl_single2,anom, template=template1)

#thresholds
thresh_alt= xr.map_blocks(thresh,anom,args=([outqt_anom]), template=anom)

#hsds
heat_days= xr.map_blocks(hotspellsfix,thresh_alt, template=anom)

heat_days= heat_days.load()
ds_heat = xr.Dataset(
data_vars=dict(
    heatdays=(["time", "latitude", "longitude"], heat_days.heatdays.values),
),
coords=dict(
    longitude=("longitude", heat_days.longitude.values),
    latitude=("latitude", heat_days.latitude.values),
    time=("time", heat_days.time.values),
),
attrs=dict(description="CanDCS hot spell data"),)

ds_heat.to_netcdf('heat2/pcicblend_tasmax_heat_days_1950-2100_fixednew.nc')

/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/xarray/core/nanops.py:127: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis=axis, dtype=dtype)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/xarray/core/nanops.py:127: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis=axis, dtype=dtype)


### RCMs

In [18]:
#read data
gfdl_esm_cmip5=xr.open_dataset('../cmip5/tas_day_GFDL-ESM2M_historical_1946-2005_regridbil.nc').sel(lat=slice(40,81)).rename({'lat':'latitude','lon':'longitude'})
gfdl_esm_wrf=xr.open_dataset('../cmip5/tas_NAM-44_NOAA-GFDL-GFDL-ESM2M_historical_r1i1p1_NCAR-WRF_v3-5_1950-2005_regridbil.nc').sel(lat=slice(40,81)).rename({'lat':'latitude','lon':'longitude'})
gfdl_esm_reg=xr.open_dataset('../cmip5/tas_NAM-44_NOAA-GFDL-GFDL-ESM2M_historical_r1i1p1_ISU-RegCM4_v4-4-rc8_1950-2005_regridbil.nc').sel(lat=slice(40,81)).rename({'lat':'latitude','lon':'longitude'})


In [26]:
#compute hotspells fixed
ddmodel_names=['gfdl-esm2m','gfdl-wrf','gfdl-regcm4']
ddmodels=[gfdl_esm_cmip5, gfdl_esm_wrf, gfdl_esm_reg]

#compute hotspells
# low resolution
for i,mod in enumerate(ddmodels):
    #prep data
    #temp_dat1=xr.open_dataset('native/tasmax_day_MBCn+PCIC-Blend_'+mod+'_historical+ssp585_regridcon.nc', chunks={'lat':10,'lon':10,'time':-1})#.load()
    temp_dat2=mod.tas.to_dataset(name='heatdays').chunk({'latitude':10,'longitude':10,'time':-1})
    try:
        temp_dat2=temp_dat2.drop_vars('height')
    except:
        pass
    # anomally
    anom=xr.map_blocks(calculate_anomaly,temp_dat2,template=temp_dat2)

    #quantiles
    template1=anom.isel(time=slice(0,365)).rename({'time':'dayofyear'})
    template1['dayofyear']=template1.dayofyear.dt.dayofyear
    #dmaxanom=cuttimestart(dmaxanom,1979)
    outqt_anom= xr.map_blocks(refqtl_single2,anom, template=template1)

    #thresholds
    thresh_alt= xr.map_blocks(thresh,anom,args=([outqt_anom]), template=anom)

    #hsds
    heat_days= xr.map_blocks(hotspells,thresh_alt, template=anom)

    heat_days= heat_days.load()
    ds_heat = xr.Dataset(
    data_vars=dict(
        heatdays=(["season", "latitude", "longitude"], heat_days.heatdays.values),
    ),
    coords=dict(
        longitude=("longitude", heat_days.longitude.values),
        latitude=("latitude", heat_days.latitude.values),
        time=("time", heat_days.time.values),
    ),
    attrs=dict(description="CanDCS hot spell data"),)
    
    ds_heat.to_netcdf('heat/'+ddmodel_names[i]+'_tasmean_heat_days_1950-2005_fixed.nc')

# for i,mod in enumerate(ddmodels):
#     temp=ddmodels[+i].tas.to_dataset(name='t2m')
#     dmaxanom=temp.groupby('time.dayofyear')-temp.groupby('time.dayofyear').mean(dim='time')
#     dmaxanom=dmaxanom.sortby(dmaxanom.time)
#     #dmaxanom=cuttimestart(dmaxanom,1979)
#     outqt_anom=refqtl_single(dmaxanom.drop_vars('dayofyear'))
#     #outqt_anom.to_netcdf('cmip6/'+mod+'_anom_quantiles_15dsmooth_1950_2023_1x1.nc')

#     thresh_anom=thresh_single(dmaxanom.drop_vars('dayofyear'),outqt_anom)
#     thresh_anom=thresh_anom.sortby(thresh_anom.time)
#     #thresh_anom.to_netcdf('cmip6/'+mod+'_anom_thresholds_5dsmooth_1950_2023_1x1.nc')

#     heat_1_d=heat_day1(thresh_anom,dmaxanom,pst=3)
#     heat_1_d.to_dataset(name='htdays').to_netcdf('heat/'+ddmodel_names[i]+'_tmax_heatdays_hist_1950_2005_regridcon.nc')